<a href="https://colab.research.google.com/github/11249a290-glitch/11249A290/blob/main/VITERBI_HMM.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from collections import defaultdict
# Extended Training Corpus with Student Additions
train_data = [
    [("The", "DET"), ("dog", "NOUN"), ("barks", "VERB")],
    [("The", "DET"), ("cat", "NOUN"), ("sleeps", "VERB")],
    [("A", "DET"), ("dog", "NOUN"), ("sees", "VERB"), ("a", "DET"), ("cat", "NOUN")],
    [("The", "DET"), ("cat", "NOUN"), ("barks", "VERB")],
    # --- 3 Student-Added Sentences Below ---
    [("He", "PRON"), ("sees", "VERB"), ("a", "DET"), ("smart", "ADJ"), ("dog", "NOUN")],
    [("The", "DET"), ("happy", "ADJ"), ("cat", "NOUN"), ("barks", "VERB")],
    [("She", "PRON"), ("likes", "VERB"), ("the", "DET"), ("dog", "NOUN")]
]

tag_count = defaultdict(int)
transition_count = defaultdict(lambda: defaultdict(int))
emission_count = defaultdict(lambda: defaultdict(int))
initial_count = defaultdict(int)                                 #11249A290
for sentence in train_data:
    prev_tag = None
    for idx, (word, tag) in enumerate(sentence):
        w_lower = word.lower()
        tag_count[tag] += 1
        emission_count[tag][w_lower] += 1
        if idx == 0:
            initial_count[tag] += 1
        else:
            transition_count[prev_tag][tag] += 1
        prev_tag = tag
tags = list(tag_count.keys())
num_sentences = len(train_data)
vocab = set(w.lower() for s in train_data for w, t in s)
V_size = len(vocab)
def initial_prob(tag):
    return (initial_count[tag] + 1) / (num_sentences + len(tags))
def transition_prob(prev_tag, tag):
    # Sum across all possible target tags to match the smoothing denominator
    total = sum(transition_count[prev_tag][t] for t in tags)
    return (transition_count[prev_tag][tag] + 1) / (total + len(tags))
def emission_prob(tag, word):
    total = tag_count[tag]
    # Smooths cleanly if the word is entirely novel (OOV)
    return (emission_count[tag][word] + 1) / (total + V_size)
def viterbi(sentence):
    words = [w.lower() for w in sentence]
    n = len(words)
    V = [dict() for _ in range(n)]
    BP = [dict() for _ in range(n)]
    for tag in tags:
        V[0][tag] = initial_prob(tag) * emission_prob(tag, words[0])
        BP[0][tag] = None
    for i in range(1, n):
        for tag in tags:
            # Clean tuple comparison based on the calculated tracking variable
            best_prob, best_prev = max(
                ((V[i-1][prev_tag] * transition_prob(prev_tag, tag), prev_tag) for prev_tag in tags),
                key=lambda x: x[0]
            )
            V[i][tag] = best_prob * emission_prob(tag, words[i])
            BP[i][tag] = best_prev
    best_last_tag = max(tags, key=lambda t: V[n-1][t])
    best_path = [best_last_tag]
    for i in range(n-1, 0, -1):
        best_path.insert(0, BP[i][best_path[0]])
    return list(zip(sentence, best_path))
test_sentences = [
    ["The", "dog", "sees", "a", "cat"],
    ["A", "cat", "barks"],
    ["The", "dog", "sleeps"],
]
for sent in test_sentences:
    result = viterbi(sent)
    print("Sentence :", " ".join(sent))
    print("Tagged   :", " ".join(f"{w}/{t}" for w, t in result))
    print("-" * 50)

Sentence : The dog sees a cat
Tagged   : The/DET dog/NOUN sees/VERB a/DET cat/NOUN
--------------------------------------------------
Sentence : A cat barks
Tagged   : A/DET cat/NOUN barks/VERB
--------------------------------------------------
Sentence : The dog sleeps
Tagged   : The/DET dog/NOUN sleeps/VERB
--------------------------------------------------


In [ ]:
from collections import defaultdict
tag_count = defaultdict(int)
transition_count = defaultdict(lambda: defaultdict(int))
emission_count = defaultdict(lambda: defaultdict(int))
initial_count = defaultdict(int)
# Count frequencies across the extended corpus
for sentence in train_data:
    prev_tag = None
    for idx, (word, tag) in enumerate(sentence):
        w_lower = word.lower()                  #11249a290
        tag_count[tag] += 1
        emission_count[tag][w_lower] += 1
        if idx == 0:
            initial_count[tag] += 1
        else:
            transition_count[prev_tag][tag] += 1
        prev_tag = tag
tags = list(tag_count.keys())
num_sentences = len(train_data)
vocab = set(w.lower() for s in train_data for w, t in s)
V_size = len(vocab)
# Print verification counts to console
print("--- ACTIVITY 5: VERIFICATION COUNTS ---")
print("Transition Counts:")
for t_from, targets in transition_count.items():
    print(f"  {t_from} -> {dict(targets)}")
print("\nEmission Counts:")
for tag, words in emission_count.items():
    print(f"  {tag} -> {dict(words)}")

--- ACTIVITY 5: VERIFICATION COUNTS ---
Transition Counts:
  DET -> {'NOUN': 6, 'ADJ': 2}
  NOUN -> {'VERB': 5}
  VERB -> {'DET': 3}
  PRON -> {'VERB': 2}
  ADJ -> {'NOUN': 2}

Emission Counts:
  DET -> {'the': 5, 'a': 3}
  NOUN -> {'dog': 4, 'cat': 4}
  VERB -> {'barks': 3, 'sleeps': 1, 'sees': 2, 'likes': 1}
  PRON -> {'he': 1, 'she': 1}
  ADJ -> {'smart': 1, 'happy': 1}


In [ ]:
def viterbi(sentence, smoothed=True):
    words = [w.lower() for w in sentence]
    n = len(words)
    # Initialize dynamic tables as a list of dictionary steps
    V = [dict() for _ in range(n)]
    BP = [dict() for _ in range(n)]
    # Trellis Initialization (Step 0)          #11249a290
    for tag in tags:
        V[0][tag] = initial_prob(tag, smoothed) * emission_prob(tag, words[0], smoothed)
        BP[0][tag] = None
    # Trellis Recursion (Steps 1 to n-1)
    for i in range(1, n):
        for tag in tags:
            best_prob, best_prev = max(
                ((V[i-1][prev_tag] * transition_prob(prev_tag, tag, smoothed), prev_tag) for prev_tag in tags),
                key=lambda x: x[0]
            )
            V[i][tag] = best_prob * emission_prob(tag, words[i], smoothed)
            BP[i][tag] = best_prev
    # Termination step
    best_last_tag = max(tags, key=lambda t: V[n-1][t])
    # Backtracking path reconstruction using the correct state indices
    best_path = [best_last_tag]
    for i in range(n-1, 0, -1):
        best_path.insert(0, BP[i][best_path[0]])
    return list(zip(sentence, best_path)), V, BP
print("--- ACTIVITY 6: RUNNING TRACE FOR 'The dog barks' ---")
test_phrase = ["The", "dog", "barks"]
result, v_table, bp_table = viterbi(test_phrase)
print("\n1. Final Tagged Output:")
print("   " + " ".join(f"{w}/{t}" for w, t in result))
print("\n2. Trellis Probability Table (V):")
for step, step_dict in enumerate(v_table):
    print(f"   Step {step} ('{test_phrase[step]}'):")
    for tag, prob in step_dict.items():
        print(f"      {tag}: {prob:.6f}")
print("\n3. Backpointer Table (BP):")
for step, step_dict in enumerate(bp_table):
    print(f"   Step {step} ('{test_phrase[step]}'): {dict(step_dict)}")
print("-" * 60)

--- ACTIVITY 6: RUNNING TRACE FOR 'The dog barks' ---

1. Final Tagged Output:
   The/DET dog/NOUN barks/VERB

2. Trellis Probability Table (V):
   Step 0 ('The'):
      DET: 0.150000
      NOUN: 0.004167
      VERB: 0.004386
      PRON: 0.017857
      ADJ: 0.005952
   Step 1 ('dog'):
      DET: 0.000577
      NOUN: 0.020192
      VERB: 0.000607
      PRON: 0.000824
      ADJ: 0.002473
   Step 2 ('barks'):
      DET: 0.000101
      NOUN: 0.000101
      VERB: 0.002551
      PRON: 0.000144
      ADJ: 0.000144

3. Backpointer Table (BP):
   Step 0 ('The'): {'DET': None, 'NOUN': None, 'VERB': None, 'PRON': None, 'ADJ': None}
   Step 1 ('dog'): {'DET': 'DET', 'NOUN': 'DET', 'VERB': 'DET', 'PRON': 'DET', 'ADJ': 'DET'}
   Step 2 ('barks'): {'DET': 'NOUN', 'NOUN': 'NOUN', 'VERB': 'NOUN', 'PRON': 'NOUN', 'ADJ': 'NOUN'}
------------------------------------------------------------


In [ ]:
test_sentences = [
    ["The", "dog", "sees", "a", "cat"],
    ["A", "cat", "barks"],
    ["The", "dog", "sleeps"],
    # --- 2 Custom Student Test Sentences Below ---
    ["She", "barks", "at", "the", "happy", "dog"],
    ["A", "smart", "cat", "sleeps"]
]
for sent in test_sentences:
    result, _, _ = viterbi(sent)
    print("Sentence :", " ".join(sent))
    print("Tagged   :", " ".join(f"{w}/{t}" for w, t in result))
    print("-" * 50)

Sentence : The dog sees a cat
Tagged   : The/DET dog/NOUN sees/VERB a/DET cat/NOUN
--------------------------------------------------
Sentence : A cat barks
Tagged   : A/DET cat/NOUN barks/VERB
--------------------------------------------------
Sentence : The dog sleeps
Tagged   : The/DET dog/NOUN sleeps/VERB
--------------------------------------------------
Sentence : She barks at the happy dog
Tagged   : She/DET barks/NOUN at/VERB the/DET happy/ADJ dog/NOUN
--------------------------------------------------
Sentence : A smart cat sleeps
Tagged   : A/DET smart/ADJ cat/NOUN sleeps/VERB
--------------------------------------------------


In [ ]:
oov_sentence = ["The", "tiny", "cat", "sleeps"]
print(f"Testing sentence with unseen word 'tiny' WITHOUT smoothing:")
try:                                                     #11249a290
    # Run the Viterbi function with smoothing disabled
    result_unsmoothed, V_unsmoothed, _ = viterbi(oov_sentence, smoothed=False)
    print("Tagged (Unsmoothed):", " ".join(f"{w}/{t}" for w, t in result_unsmoothed))
except Exception as e:
    print(f"Algorithm failed with exception: {type(e).__name__} -> {e}")

Testing sentence with unseen word 'tiny' WITHOUT smoothing:
Tagged (Unsmoothed): The/DET tiny/DET cat/DET sleeps/DET


In [ ]:
print("\n--- ACTIVITY 9: COMPARISON WITH NLTK ---")
try:                        #11249a290
    import nltk
    # Secure clean local access to standard tools
    try:
        nltk.data.find('taggers/averaged_perceptron_tagger_eng')
    except LookupError:
        nltk.download('averaged_perceptron_tagger_eng', quiet=True)
    nltk_sentence = ["The", "dog", "sees", "a", "cat"]
    print("NLTK Tagging Output for ['The', 'dog', 'sees', 'a', 'cat']:")
    print(nltk.pos_tag(nltk_sentence))
except ImportError:
    print("NLTK is not installed. Please run 'pip install nltk' in your terminal.")


--- ACTIVITY 9: COMPARISON WITH NLTK ---
NLTK Tagging Output for ['The', 'dog', 'sees', 'a', 'cat']:
[('The', 'DT'), ('dog', 'NN'), ('sees', 'VBZ'), ('a', 'DT'), ('cat', 'NN')]


In [ ]:
print("\n--- ACTIVITY 8: SMOOTHING REMOVAL EFFECT ---")
oov_sentence = ["The", "tiny", "cat", "sleeps"]
print(f"Testing sentence with unseen word 'tiny' WITHOUT smoothing:")
try:
    # Run the Viterbi function with smoothing disabled
    result_unsmoothed, V_unsmoothed, _ = viterbi(oov_sentence, smoothed=False)
    print("Tagged (Unsmoothed):", " ".join(f"{w}/{t}" for w, t in result_unsmoothed))
except Exception as e:
    print(f"Algorithm failed with exception: {type(e).__name__} -> {e}")


--- ACTIVITY 8: SMOOTHING REMOVAL EFFECT ---
Testing sentence with unseen word 'tiny' WITHOUT smoothing:
Tagged (Unsmoothed): The/DET tiny/DET cat/DET sleeps/DET
